<a href="https://colab.research.google.com/github/WingmanShi/WingmanShi-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

First, i gonna look at the each table.

Prompt: "give me the code: for each table print the table name, the number of rows and the first three records"

I'm expecting to see the table name, total row count, and the first 3 rows formatted nicely for each of the six tables.

In [4]:
from IPython.display import display

# I'm expecting to see the table name, total row count, and the first 3 rows formatted nicely for each of the six tables.
for name, df in tables.items():
    print(f"--- Table: {name} ({df.shape[0]:,} rows) ---")
    display(df.head(3))
    print("\n")

--- Table: orders (14,494 rows) ---


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00




--- Table: order_status_events (56,812 rows) ---


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z




--- Table: order_lines (36,120 rows) ---


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5


... (2 outputs left out)



--- Table: restaurants (67 rows) ---


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01




--- Table: customers (4,200 rows) ---


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds


Next, i gonna check for missing values.

Prompt: "now, let's cheking the missing values."

I'm expecting to see a summary count of missing (null) values for every column across all six tables in our dataset.

In [5]:
for name, df in tables.items():
    missing_counts = df.isnull().sum()
    print(f"--- Missing values in: {name} ---")
    print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "No missing values found.")
    print("\n")

--- Missing values in: orders ---
No missing values found.


--- Missing values in: order_status_events ---
No missing values found.


--- Missing values in: order_lines ---
No missing values found.


--- Missing values in: payments ---
No missing values found.


--- Missing values in: restaurants ---
No missing values found.


--- Missing values in: customers ---
No missing values found.




Great! there is no any missing values.

Let's check if the primary key is unique in each table.

Prompt: "Let's check if the primary key is unique in each table. show me for each table the number of rows and teh number of unique ids. the unique ids should be order_id for order; event_id for order_status_events; order_id for order_lines; payment_id for payments; restaurant_id for restaurant and customer_id for customers"

I'm expecting to see whether each table's primary key column contains unique values (and the count of any duplicates if they exist).

In [7]:
# Define the identifier column to check for each table
id_columns = {
    "orders": "order_id",
    "order_status_events": "event_id",
    "order_lines": "order_id",         # Note: order_id appears multiple times per order here (one per line)
    "payments": "payment_id",
    "restaurants": "restaurant_id",
    "customers": "customer_id",
}

for name, id_col in id_columns.items():
    df = tables[name]
    n_rows = len(df)
    n_unique = df[id_col].nunique()
    print(f"Table: {name}")
    print(f"  - Total rows: {n_rows:,}")
    print(f"  - Unique '{id_col}': {n_unique:,}")
    if n_rows == n_unique:
        print("  - Status: UNIQUE (1:1 match with row count)\n")
    else:
        print(f"  - Status: NOT UNIQUE ({n_rows - n_unique:,} difference/duplicates)\n")

Table: orders
  - Total rows: 14,494
  - Unique 'order_id': 14,494
  - Status: UNIQUE (1:1 match with row count)

Table: order_status_events
  - Total rows: 56,812
  - Unique 'event_id': 56,812
  - Status: UNIQUE (1:1 match with row count)

Table: order_lines
  - Total rows: 36,120
  - Unique 'order_id': 14,494
  - Status: NOT UNIQUE (21,626 difference/duplicates)

Table: payments
  - Total rows: 16,423
  - Unique 'payment_id': 16,393
  - Status: NOT UNIQUE (30 difference/duplicates)

Table: restaurants
  - Total rows: 67
  - Unique 'restaurant_id': 64
  - Status: NOT UNIQUE (3 difference/duplicates)

Table: customers
  - Total rows: 4,200
  - Unique 'customer_id': 4,200
  - Status: UNIQUE (1:1 match with row count)



As you can see, primary key in order_line, payments and restaurants is not unique. Let's look at these each table.

Let's check for restaurant first.

Prompt: "find items in restaurants that share the same restaurant_id. i.e. if restaurant_id = r001 is in the table twice, print both these rows. "

I'm expecting to find and display the exact rows that share duplicate primary/identifier keys in the restaurants table.

In [8]:
# Restaurants duplicate check
rest_df = tables["restaurants"]
duplicate_restaurants = rest_df[rest_df.duplicated(subset=["restaurant_id"], keep=False)]
display(duplicate_restaurants.sort_values(by="restaurant_id"))

,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27
48,R040,Corner Vegan 40,Vegan,London,2025-01-01


The restaurant_id column alone is not unique because restaurants can change their details (like moving to a new city) over time, resulting in historical or versioned records. So the actual composite primary key is a combination of restaurant_id and effective_from. For example, restaurant R007 appears twice because it operated in Bristol starting 2025-01-01 and later opened or moved to London on 2026-04-14.

now, let's check for payments.

Prompt: "find items in payments that share the same payment_id."

I'm expecting to find and display the exact rows that share duplicate payment IDs in the payments table.

In [9]:
# Payments duplicate check
pay_df = tables["payments"]
duplicate_payments = pay_df[pay_df.duplicated(subset=["payment_id"], keep=False)]
display(duplicate_payments.sort_values(by="payment_id"))

,payment_id,order_id,payment_type,amount,paid_at
5949,P000219,O000185,charge,25.49,2026-03-02 19:05:43
13373,P000219,O000185,charge,25.49,2026-03-02 19:05:43
1673,P000927,O000815,charge,23.99,2026-03-08 11:53:53
4665,P000927,O000815,charge,23.99,2026-03-08 11:53:53
6582,P000998,O000874,charge,50.99,2026-03-08 19:19:26
3747,P000998,O000874,charge,50.99,2026-03-08 19:19:26
975,P001859,O001631,charge,38.49,2026-03-14 22:43:50
5224,P001859,O001631,charge,38.49,2026-03-14 22:43:50
15431,P002189,O001926,charge,41.99,2026-03-17 18:22:24
9230,P002189,O001926,charge,41.99,2026-03-17 18:22:24


The duplicate payment_id rows are exact identical row duplicates (same amount, timestamp, order ID, and payment type appearing twice in the raw file). payment_id is intended to be the unique identifier for a transaction, but the raw data ingestion contains duplicate log entries or retried gateway logs that need to be cleaned (deduplicated) by dropping exact row matches before building your silver/gold tables.

now, let's check for order_lines.

Prompt: "find items in order_lines that share the same order_id."

I'm expecting to find and display the exact rows in order_lines the primary key (order_id) has duplicate entries.

In [11]:
# Order lines duplicate check (using composite key: order_id and line_no)
lines_df = tables["order_lines"]
duplicate_lines = lines_df[lines_df.duplicated(subset=["order_id"], keep=False)]
display(duplicate_lines.sort_values(by=["order_id"]))

,order_id,line_no,item_name,quantity,unit_price
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0
5,O000003,2,Soft drink,1,2.0
...,...,...,...,...,...
36116,O014493,2,Falafel wrap,2,6.5
36115,O014493,1,Falafel wrap,2,6.5
36117,O014493,3,Fried rice,2,7.5
36118,O014494,1,Burrito,3,8.0


Because an order naturally contains multiple line items (e.g., a drink, a side, and a main dish), order_id will repeat across multiple rows. So, the true primary key for order_lines is a composite key consisting of order_id and line_no.

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

**Grain of the Final Table :** One row per Week, Cuisine, and City combination (aggregating performance metrics across these dimensions).

**Decisions**

- **week:** Based on the order placement timestamp (order_ts), truncated to the start of the week (e.g., Monday).

- **Order:** Only orders that successfully complete the lifecycle (reaching a delivered/completed status via order_status_events) will count as valid orders. Cancelled or abandoned orders will be excluded from the main net revenue and delivery time calculations (though cancellation tracking can be noted if time permits).

- **Net revenue:** Net revenue is calculated as the sum of item line prices multiplied by their quantities (unit_price * quantity from order_lines), plus the delivery_fee from the orders table, minus any refund amounts found in the payments table.

- **Delivery time:** Delivery time is the duration in minutes between the initial order placement event (placed status in order_status_events) and the final delivery event (delivered status). It can only be measured for orders that contain both timestamp events.

- **Handling Data Quality Issues:**

   - Restaurants: Account for historical changes/versioning by joining orders to restaurants using both restaurant_id and ensuring the restaurant was effective at the time of the order_ts.

   - Payments: Deduplicate exact row matches in the payments table before aggregating to prevent inflated revenue figures.

**Plan for Building**

**Silver Layer - Clean Orders** (df_orders_clean):

Filter valid orders from orders. [ = rows in orders]  (minus any invalid/orphaned records).

**Silver Layer - Clean Order Lines** (df_order_lines_clean):

Aggregate or process order_lines to compute total item value per order. One row per order ID or aggregated back to match unique order grain.

**Silver Layer - Restaurant History** (df_restaurants_clean):

Handle the versioned restaurants table by matching the active date window (effective_from). [ = rows in restaurants].

**Silver Layer - Delivery Times** (df_delivery_times):

Pivot or filter order_status_events to compute duration between placed and delivered. Expected row counts less than or equal to rows in orders (only successful deliveries).

**Gold Reporting Table** (df_gold_weekly_ops):

Join the cleaned silver components together, group by week, cuisine, and city, and calculate metrics (number of orders, net revenue, average delivery time). Expected row counts: Number of unique weeks $\times$ unique cuisines $\times$ unique cities (significantly smaller than base tables).


## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Now, i am cleaning and preparing the orders table. I will filter and transform the base orders table to set up our primary order-level grain, including extracting the weekly time bucket.

Prompt: "Give me the code to clean the orders table, parse the order timestamp as a UTC datetime, and extract the start of the week for each order"

I expect to see the shape of the cleaned orders table and its first few rows with the newly added week column.

In [12]:
# Step 1: Build Silver Layer - Clean Orders
df_orders = tables["orders"].copy()

# Convert order timestamp to datetime
df_orders["order_ts"] = pd.to_datetime(df_orders["order_ts"])

# Extract the start of the week (Monday) based on order_ts
df_orders["order_week"] = df_orders["order_ts"].dt.to_period("W-MON").dt.start_time.dt.date

print(f"Cleaned orders table shape: {df_orders.shape}")
display(df_orders.head(3))

Cleaned orders table shape: (14494, 6)


/tmp/ipykernel_2796/607873202.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_orders["order_week"] = df_orders["order_ts"].dt.to_period("W-MON").dt.start_time.dt.date


,order_id,customer_id,restaurant_id,order_ts,delivery_fee,order_week
0,O000001,C01161,R052,2026-03-01 09:13:23+00:00,1.49,2026-02-24
1,O000002,C00723,R027,2026-03-01 09:15:33+00:00,2.49,2026-02-24
2,O000003,C00781,R044,2026-03-01 09:23:42+00:00,0.00,2026-02-24


Next, I am cleaning and aggregating the order_lines table to compute the total item value and item count for each order ID.

Prompt: "Give me the code to aggregate order_lines by order_id, calculating the total item value (quantity * unit_price) and total item count for each order"

I expect to see the shape of the aggregated order lines table and its first few rows matching the order grain.

In [13]:
# Step 2: Build Silver Layer - Clean and Aggregate Order Lines
df_lines = tables["order_lines"].copy()

# Calculate total item line value
df_lines["line_total"] = df_lines["quantity"] * df_lines["unit_price"]

# Aggregate to order grain
df_order_lines_agg = df_lines.groupby("order_id").agg(
    total_item_value=("line_total", "sum"),
    total_items=("quantity", "sum")
).reset_index()

print(f"Aggregated order lines table shape: {df_order_lines_agg.shape}")
display(df_order_lines_agg.head(3))

Aggregated order lines table shape: (14494, 3)


,order_id,total_item_value,total_items
0,O000001,36.0,3
1,O000002,29.5,5
2,O000003,12.5,3


Let's build the code to handle the versioned restaurants table and map each order to the correct restaurant details (cuisine and city) active at the time of the order (order_ts).

prompt: "Give me the code to join the orders table with the versioned restaurants table, ensuring that the restaurant's effective_from date is less than or equal to the order timestamp, and selecting the most recent version if multiple exist"

I expect to see the joined table shape containing the cuisine and city columns for each order.

In [29]:
# Step 3 : Build Silver Layer - Restaurant History Mapping using Cleaned Restaurants
# Sort cleaned restaurants by effective_from to enable proper merging/asof joining
df_restaurants_clean_sorted = df_restaurants_clean.sort_values("effective_from")

# Use merge_asof to find the correct restaurant version active at order_ts
df_orders_sorted = df_orders.sort_values("order_ts")
df_orders_rest = pd.merge_asof(
    df_orders_sorted,
    df_restaurants_clean_sorted,
    left_on="order_ts",
    right_on="effective_from",
    by="restaurant_id",
    direction="backward"
)

print(f"Orders with cleaned restaurant history shape: {df_orders_rest.shape}")
display(df_orders_rest[["order_id", "restaurant_id", "restaurant_name", "cuisine", "city"]].head(3))

Orders with cleaned restaurant history shape: (14494, 10)


,order_id,restaurant_id,restaurant_name,cuisine,city
0,O000001,R052,Old Burgers 52,Burgers,London
1,O000002,R027,Corner Chinese 27,Chinese,London
2,O000003,R044,Fresh Burgers 44,Burgers,Birmingham


I will process order_status_events to compute the delivery duration in minutes for each successful order (time from placed to delivered).

Prompt: "Give me the code to pivot or filter order_status_events to extract the placed timestamp and delivered timestamp for each order, and calculate the delivery duration in minutes"

I expect to see a dataframe containing order IDs and their calculated delivery times in minutes.

In [25]:
# Step 4: Build Silver Layer - Delivery Times Calculation
df_events = tables["order_status_events"].copy()
df_events["event_ts"] = pd.to_datetime(df_events["event_ts"])

# Filter for 'placed' and 'delivered' statuses
placed_events = df_events[df_events["status"] == "placed"][["order_id", "event_ts"]].rename(columns={"event_ts": "placed_ts"})
delivered_events = df_events[df_events["status"] == "delivered"][["order_id", "event_ts"]].rename(columns={"event_ts": "delivered_ts"})

# Merge them together to find the duration for successful orders
df_delivery = pd.merge(placed_events, delivered_events, on="order_id", how="inner")
df_delivery["delivery_time_mins"] = (df_delivery["delivered_ts"] - df_delivery["placed_ts"]).dt.total_seconds() / 60.0

print(f"Delivery times table shape: {df_delivery.shape}")
display(df_delivery.head(3))

Delivery times table shape: (13549, 4)


,order_id,placed_ts,delivered_ts,delivery_time_mins
0,O004658,2026-04-08 19:11:30+00:00,2026-04-08 19:53:04+00:00,41.566667
1,O004658,2026-04-08 19:11:30+00:00,2026-04-08 19:54:25+00:00,42.916667
2,O001338,2026-03-12 20:23:31+00:00,2026-03-12 20:56:18+00:00,32.783333


I will deduplicate the payments table to remove exact duplicate log entries before computing the financial metrics.

Prompt: "Give me the code to drop exact row duplicates from the payments table to ensure each payment ID is unique"

I expect to see the reduced row count of the cleaned payments table matching the unique payment ID count.

In [26]:
# Step 5: Build Silver Layer - Clean Payments
df_payments = tables["payments"].copy()

# Drop exact row duplicates
df_payments_clean = df_payments.drop_duplicates()

print(f"Original payments rows: {len(df_payments):,}")
print(f"Cleaned payments rows: {len(df_payments_clean):,}")

Original payments rows: 16,423
Cleaned payments rows: 16,393


Now, I will make sure the city column in the restaurants table is properly cleaned by stripping any leading or trailing whitespaces as well as standardizing the casing.

Prompt: "Give me the code to strip whitespaces and properly capitalize the city column in the restaurants table"

I expect to see the restaurants cities cleaned with all extra spaces removed and correct casing applied.

In [30]:
# Silver Layer: Clean and standardize restaurants table city (strip whitespaces and capitalize)
df_restaurants_clean = df_restaurants.copy()
df_restaurants_clean["city"] = df_restaurants_clean["city"].astype(str).str.strip().str.capitalize()

print(f"Cleaned restaurants cities (unique): {df_restaurants_clean['city'].unique()}")

Cleaned restaurants cities (unique): ['London' 'Manchester' 'Birmingham' 'Leeds' 'Bristol']


Now, I will combine all the cleaned silver components together, join them with the aggregated order lines and payment charges/refunds, and group by order_week, cuisine, and city to create the final reporting table.

Prompt: "Give me the code to join orders, order lines, delivery times, payments, and restaurant details, then group by order_week, cuisine, and city to calculate total orders, net revenue, and average delivery time"

I expect to see our final gold summary table showing weekly operations performance grouped by week, cuisine, and city.

In [31]:
# Step 6: Build Gold Reporting Table
# Aggregate payments by order_id (summing charges and refunds)
df_payments_agg = df_payments_clean.groupby("order_id")["amount"].sum().reset_index().rename(columns={"amount": "total_paid"})

# Merge everything into the base orders restaurant dataset
df_gold = df_orders_rest.merge(df_order_lines_agg, on="order_id", how="left")
df_gold = df_gold.merge(df_delivery[["order_id", "delivery_time_mins"]], on="order_id", how="left")
df_gold = df_gold.merge(df_payments_agg, on="order_id", how="left")

# Calculate net revenue per order: item value + delivery fee
df_gold["net_revenue"] = df_gold["total_item_value"].fillna(0) + df_gold["delivery_fee"].fillna(0)

# Group by week, cuisine, and city
df_gold_weekly_ops = df_gold.groupby(["order_week", "cuisine", "city"]).agg(
    n_orders=("order_id", "count"),
    net_revenue=("net_revenue", "sum"),
    avg_delivery_time_mins=("delivery_time_mins", "mean")
).reset_index()

print(f"Gold table shape: {df_gold_weekly_ops.shape}")
display(df_gold_weekly_ops.head(10))

Gold table shape: (694, 6)


,order_week,cuisine,city,n_orders,net_revenue,avg_delivery_time_mins
0,2026-02-24,Burgers,Birmingham,7,270.96,35.735714
1,2026-02-24,Burgers,Bristol,4,105.46,40.200000
2,2026-02-24,Burgers,Leeds,4,136.47,42.075000
3,2026-02-24,Burgers,London,20,844.33,36.837963
4,2026-02-24,Burgers,London,3,85.98,52.700000
5,2026-02-24,Chinese,Birmingham,13,401.90,36.259091
6,2026-02-24,Chinese,London,17,589.85,36.497778
7,2026-02-24,Chinese,Manchester,1,24.99,33.666667
8,2026-02-24,Indian,Birmingham,2,72.48,40.366667
9,2026-02-24,Indian,London,15,609.89,36.703571


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

- I did not verify whether orders with multiple payment gateway log entries or retried charges had their amounts correctly totaled or if they caused double-counting in our net revenue metric.

- I could not verify if cancelled or abandoned orders were completely excluded from the delivery time averages or if missing delivery events skewed the duration calculations.

- I did not verify whether there were orphaned records in order_lines or order_status_events referencing order IDs that do not exist in the primary orders table.

- I could not verify if the refund amounts found in the payments table were properly subtracted from net revenue or if they were only accounted for via raw charges.

- I did not verify whether any restaurants had overlapping active date ranges (effective_from) that could result in ambiguous historical matches during the merge_asof join.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.